# 04 - Parameter Losses and Gaussian Matching

Curve losses are permutation invariant by construction, but the direct
parameter losses (`param_l1`, `param_huber`) compare per-Gaussian triplets
and therefore depend on the ordering of the predicted Gaussians relative to
the ground truth. The `ParamMatcher` with strategy `sort_gt_by_mu` reorders
the ground-truth Gaussians by their centre before the comparison. This
notebook confirms (a) the parameter loss falls to zero only when prediction
and target agree, and (b) the matcher makes the loss invariant to a
permutation of the ground-truth components.

Modules exercised: `Loss._param_term`, `ParamMatcher.match_torch`,
`LossComponents.param_l1`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy             as np
import torch
import matplotlib.pyplot as plt

SEED   = 0
DEVICE = torch.device("cpu")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

plt.rcParams.update({
    "figure.dpi"     : 120,
    "savefig.dpi"    : 300,
    "font.size"      : 10,
    "axes.grid"      : True,
    "grid.alpha"     : 0.3,
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
})

print("repo root :", REPO_ROOT)
print("torch     :", torch.__version__)
print("device    :", DEVICE)


In [ ]:
from configuration.training_config import GaussianConfig, GeometryConfig, LossConfig
from pipelines.backbone_pipeline.loss import Loss, ParamMatcher
from tools import NullLogger, NullTracker

class IdentityNormStats:
    def normalize_output(self, tensor):
        return tensor

    def denormalize_output(self, tensor):
        return tensor

X_MIN, X_MAX = -10.0, 40.0
x_axis       = torch.linspace(X_MIN, X_MAX, 48, dtype=torch.float32)
gaussian_cfg = GaussianConfig(n_default_gaussians=2, x_min=X_MIN, x_max=X_MAX)
loss_cfg     = LossConfig(use_param_l1=True, weight_param_l1=1.0, param_match='sort_gt_by_mu')

criterion = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg,
                 loss_cfg, IdentityNormStats(), GeometryConfig())

def make_params(triplets, H=2, W=2):
    flat = torch.tensor([v for t in triplets for v in t], dtype=torch.float32)
    return flat.reshape(1, len(flat), 1, 1).expand(1, len(flat), H, W).contiguous()


## Parameter L1 vs a centre error

We hold the target fixed and move one predicted centre away from its true
value. The per-parameter L1 should grow linearly with the error and reach a
minimum when prediction equals target.

In [ ]:
gt = make_params([(3.0, 0.0, 2.0), (1.5, 20.0, 6.0)])

offsets = torch.linspace(-15.0, 15.0, 61)
vals = []
for off in offsets:
    pred = make_params([(3.0, float(off), 2.0), (1.5, 20.0, 6.0)])
    out  = criterion(pred, gt)
    vals.append(float(out['components']['param_l1']))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(offsets.numpy(), vals, color='C0')
ax.axvline(0.0, color='grey', ls=':', lw=1, label='true centre')
ax.set_xlabel('predicted centre of Gaussian 1 [m]')
ax.set_ylabel('param_l1 loss')
ax.set_title('Parameter L1 vs centre error')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## Permutation invariance from `sort_gt_by_mu`

We compare a fixed prediction against the target and against the same target
with its two Gaussians swapped. With matching enabled the loss should be
identical; with the matcher set to a no-op strategy the swapped target
produces a different, larger loss.

In [ ]:
pred = make_params([(3.0, 0.0, 2.0), (1.5, 20.0, 6.0)])
gt_a = make_params([(3.0, 0.0, 2.0), (1.5, 20.0, 6.0)])
gt_b = make_params([(1.5, 20.0, 6.0), (3.0, 0.0, 2.0)])

def param_loss(cfg_match):
    cfg = LossConfig(use_param_l1=True, weight_param_l1=1.0, param_match=cfg_match)
    crit = Loss(x_axis, NullLogger(), NullTracker(), gaussian_cfg, cfg, IdentityNormStats(), GeometryConfig())
    return float(crit(pred, gt_a)['components']['param_l1']), float(crit(pred, gt_b)['components']['param_l1'])

sorted_aligned, sorted_swapped = param_loss('sort_gt_by_mu')
none_aligned,   none_swapped   = param_loss('none')

labels = ['aligned GT', 'swapped GT']
fig, ax = plt.subplots(figsize=(6, 4))
width = 0.35
idx = np.arange(2)
ax.bar(idx - width / 2, [sorted_aligned, sorted_swapped], width, label='match = sort_gt_by_mu')
ax.bar(idx + width / 2, [none_aligned,   none_swapped],   width, label='match = none')
ax.set_xticks(idx); ax.set_xticklabels(labels)
ax.set_ylabel('param_l1 loss')
ax.set_title('Effect of ground-truth ordering')
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

print('sort_gt_by_mu : aligned=%.5f  swapped=%.5f' % (sorted_aligned, sorted_swapped))
print('none          : aligned=%.5f  swapped=%.5f' % (none_aligned, none_swapped))


## Expected visual outcome

The parameter L1 curve is a clean V with its minimum at the true centre.
In the bar chart, the `sort_gt_by_mu` matcher yields equal loss for the
aligned and swapped ground truth (both small), confirming permutation
invariance, while the `none` strategy gives a large loss for the swapped
ordering because it compares mismatched Gaussians.